In [1]:
!pip install playwright nest-asyncio pandas openpyxl
!playwright install chromium

(node:28493) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
159.6 MiB [                    ] 0% 0.0s159.6 MiB [                    ] 0% 688.6s159.6 MiB [                    ] 0% 522.3s159.6 MiB [                    ] 0% 452.9s159.6 MiB [                    ] 0% 448.0s159.6 MiB [                    ] 0% 413.2s159.6 MiB [                    ] 0% 394.8s159.6 MiB [                    ] 0% 538.1s159.6 MiB [                    ] 0% 438.8s159.6 MiB [                    ] 0% 382.3s159.6 MiB [                    ] 0% 328.4s159.6 MiB [                    ] 0% 300.5s159.6 MiB [                    ] 0% 282.0s159.6 MiB [                    ] 0% 265.1s159.6 MiB [                    ] 0% 244.5s159.6 MiB [                    ] 0% 217.4s159.6 MiB [                  

In [2]:
import asyncio, json
from playwright.async_api import async_playwright
import nest_asyncio
nest_asyncio.apply()

COOKIES_FILE = "cookies.json"

async def save_cookies():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()
        await page.goto("https://vn.trip.com/")

        print("\u27a1\ufe0f H\u00e3y \u0111\u0103ng nh\u1eadp th\u1ee7 c\u00f4ng (Email/Google/Apple).")
        print("\u23f3 B\u1ea1n c\u00f3 60 gi\u00e2y \u0111\u1ec3 login...")
        await asyncio.sleep(60)

        cookies = await context.cookies()
        with open(COOKIES_FILE, "w", encoding="utf-8") as f:
            json.dump(cookies, f)

        print(f"\u2705 Cookies \u0111\u00e3 \u0111\u01b0\u1ee3c l\u01b0u v\u00e0o {COOKIES_FILE}")
        await browser.close()

await save_cookies()

➡️ Hãy đăng nhập thủ công (Email/Google/Apple).
⏳ Bạn có 60 giây để login...
✅ Cookies đã được lưu vào cookies.json


### Main code

In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
import random
import time
import os
import re
import json
from datetime import datetime, timedelta
from playwright.async_api import async_playwright
from urllib.parse import quote
from openpyxl import load_workbook

nest_asyncio.apply()

# Unicode chars for print formatting (avoid backslash in f-string expressions)
_LINE = '─'
_CHECK = '✅'
_WARN = '⚠️'
_TICK = '✓'
_CROSS = '✗'
_BAN = '🚫'

# ============================================================
# CONFIG
# ============================================================
COOKIES_FILE = "cookies.json"

# Google Sheet config
GOOGLE_SHEET_ID = ""  # Để trống = đọc từ file local INPUT_FILE
GOOGLE_SHEET_NAME = "Hotel Link"

# Local file config (xlsx với hyperlink hoặc csv)
INPUT_FILE = "./tr4.csv"
INPUT_SHEET_NAME = "Hotel Link"  # Chỉ dùng cho xlsx
TEMP_OUTPUT_FILE = "tripcom_prices_temp_HN2.csv"
OUTPUT_PREFIX = "tripcom_prices_"

HEADLESS = True
DEBUG_SCREENSHOTS = False

CHECKIN_OFFSET = 1          # Số ngày cộng thêm kể từ ngày crawl (ví dụ: today + 3)
NUM_WORKERS = 6
WEEKS_PER_HOTEL = 7        # Max weeks crawled in parallel per hotel
DAYS_PER_WEEK = 7           # Số ngày thử trong mỗi tuần (fallback từng ngày)
RETRIES_PER_DAY = 2
PAGE_TIMEOUT = 30000
BATCH_SIZE = 10
MAX_RETRY_ROUNDS = 1
TARGET_NA_RATE = 0.10

AUTO_RETRY_NA_SOLDOUT = True

DELAY_RANGE = (1.5, 3.0)
HOTEL_DELAY = (2, 4)
RETRY_COOL_DOWN = (5, 10)
RETRY_PAGE_TIMEOUT = [30000, 45000]

USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
]

STEALTH_SCRIPT = """
Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
window.chrome = { runtime: {} };
const originalQuery = window.navigator.permissions.query;
window.navigator.permissions.query = (parameters) => (
    parameters.name === 'notifications'
    ? Promise.resolve({state: Notification.permission})
    : originalQuery(parameters)
);
"""

# Trip.com price extraction JS — adapted for Trip.com selectors
EXTRACT_PRICES_JS = """(targetRoom) => {
    const results = [];
    // Trip.com room cards: tìm bằng selector chứa 'commonRoomCard' trong class
    const roomCards = document.querySelectorAll("div[class*='commonRoomCard__']");
    const targetLower = targetRoom.toLowerCase().trim();

    function extractVNDPrice(text) {
        // "Total (incl. taxes & fees): VND 2,558,981" -> "VND 2,558,981"
        // "Incl. taxes & fees" -> null (no number found)
        const m1 = text.match(/VND\s*[\d,]+/i);
        if (m1) { const c = m1[0].trim(); if (/\d/.test(c)) return c; }
        const m2 = text.match(/[\d,]+\s*VND/i);
        if (m2) { const c = m2[0].trim(); if (/\d/.test(c)) return c; }
        const m3 = text.match(/\u20ab\s*[\d,]+|[\d,]+\s*\u20ab/);
        if (m3) { const c = m3[0].trim(); if (/\d/.test(c)) return c; }
        const m4 = text.match(/[\d,]{7,}/);
        if (m4) return m4[0].trim();
        return null;
    }

    const bodyText = document.body.innerText || '';
    const soldOut = /sold\s*out|h\u1ebft\s*ph\u00f2ng|no\s*room/i.test(bodyText) && roomCards.length === 0;

    if (soldOut && roomCards.length === 0) {
        return {found: false, soldOut: true, soldOutType: 'hotel', allRooms: 0, pageTitle: document.title, bodySnippet: bodyText.substring(0, 300)};
    }

    roomCards.forEach(card => {
        // Room title
        const titleEl = card.querySelector("span[class*='commonRoomCard-title']");
        const name = titleEl ? titleEl.textContent.trim() : '';
        const cardText = card.innerText || '';

        // Prices: ưu tiên Total (incl. taxes) > display price
        let prices = [];

        // 1) Total price (incl. taxes & fees) — extract only VND number
        card.querySelectorAll("div[class*='priceExplain']").forEach(el => {
            const txt = el.textContent.trim();
            const price = extractVNDPrice(txt);
            if (price) prices.push(price);
        });

        // 2) Display price — extract only VND number
        if (prices.length === 0) {
            card.querySelectorAll("div[class*='displayPrice'] span, span[class*='displayPrice']").forEach(el => {
                const txt = el.textContent.trim();
                const price = extractVNDPrice(txt);
                if (price) prices.push(price);
            });
        }

        // 3) Fallback: any price-like text in card
        if (prices.length === 0) {
            const walker = document.createTreeWalker(card, NodeFilter.SHOW_TEXT);
            while (walker.nextNode()) {
                const price = extractVNDPrice(walker.currentNode.textContent.trim());
                if (price) prices.push(price);
            }
        }

        // Sold out check per room
        let soldOutPrice = null;
        const soMatch = cardText.match(/sold\s*out|h\u1ebft\s*ph\u00f2ng/i);
        if (soMatch && prices.length === 0) {
            soldOutPrice = extractVNDPrice(cardText);
        }

        results.push({
            name, nameLower: name.toLowerCase().trim(),
            prices: prices.slice(0, 5),
            matched: name.toLowerCase().trim().includes(targetLower) || targetLower.includes(name.toLowerCase().trim()),
            soldOutPrice
        });
    });

    // Find target room
    const target = results.find(r => r.matched);
    if (target) {
        if (target.prices.length > 0) return {found: true, price: target.prices[0], room: target.name, allRooms: results.length};
        if (target.soldOutPrice) return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: target.soldOutPrice, room: target.name, allRooms: results.length};
    }

    // Partial match
    const partial = results.find(r => r.nameLower.includes(targetLower) || targetLower.includes(r.nameLower));
    if (partial) {
        if (partial.prices.length > 0) return {found: true, price: partial.prices[0], room: partial.name, allRooms: results.length, partial: true};
        if (partial.soldOutPrice) return {found: false, soldOut: true, soldOutType: 'room', soldOutPrice: partial.soldOutPrice, room: partial.name, allRooms: results.length, partial: true};
    }

    // All sold out
    const allSoldOut = results.length > 0 && results.every(r => r.soldOutPrice && r.prices.length === 0);
    if (allSoldOut) {
        const rel = target || partial || results[0];
        return {found: false, soldOut: true, soldOutType: 'all_rooms', soldOutPrice: rel.soldOutPrice, allRooms: results.length};
    }

    return {found: false, soldOut: false, allRooms: results.length, roomNames: results.map(r => r.name), pageTitle: document.title, bodySnippet: (document.body.innerText || '').substring(0, 300)};
}"""

# ============================================================
# HELPERS
# ============================================================
def read_hotels_from_source():
    if GOOGLE_SHEET_ID:
        return read_hotels_from_gsheet(GOOGLE_SHEET_ID, GOOGLE_SHEET_NAME)
    elif INPUT_FILE.endswith('.xlsx'):
        return read_hotels_from_xlsx(INPUT_FILE, INPUT_SHEET_NAME)
    else:
        return read_hotels_from_csv(INPUT_FILE)

def read_hotels_from_gsheet(sheet_id, sheet_name=""):
    try:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv"
        if sheet_name:
            url += f"&sheet={quote(sheet_name)}"
        print(f"\U0001f4e1 Đọc Google Sheet: ...{sheet_id[-8:]} | tab: {sheet_name or '(default)'}", flush=True)
        df = pd.read_csv(url)
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        if not all(col in df.columns for col in required_cols):
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        print(f"\u2705 {len(df)} hotels từ Google Sheet", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"\u274c Lỗi đọc Google Sheet: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_xlsx(file_path, sheet_name):
    """Đọc hotel list từ xlsx với hyperlink (giữ logic gốc của Trip.com)."""
    try:
        wb = load_workbook(filename=file_path, data_only=True)
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows(min_row=2, max_col=2):
            hotel_cell = row[0]
            room_cell = row[1]
            hotel_name = hotel_cell.value or ""
            if hotel_cell.hyperlink:
                hotel_url = hotel_cell.hyperlink.target
            else:
                hotel_url = hotel_cell.value if hotel_cell.value else ""
            room_type = room_cell.value if room_cell else ""
            # Chuyển vn.trip.com -> trip.com
            if isinstance(hotel_url, str):
                hotel_url = hotel_url.replace("vn.trip.com", "trip.com")
            rows.append({'hotel_name': hotel_name, 'hotel_url': hotel_url, 'room_type': room_type})
        df = pd.DataFrame(rows)
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '') & df['hotel_url'].str.startswith('http')]
        print(f"\u2705 {len(df)} hotels từ {file_path} (sheet: {sheet_name})", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"\u274c Lỗi đọc xlsx: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        required_cols = ['hotel_name', 'hotel_url', 'room_type']
        if not all(col in df.columns for col in required_cols):
            df.columns = ['hotel_name', 'hotel_url', 'room_type'] + list(df.columns[3:])
        df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
        print(f"\u2705 {len(df)} hotels từ {file_path}", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"\u274c Lỗi đọc CSV: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def save_backup_csv(all_week_prices, filename):
    """Save CSV — only-improve (không ghi NA lên giá thật) + atomic write."""
    try:
        rows = []
        written_keys = set()
        if os.path.exists(filename):
            try:
                df_old = pd.read_csv(filename, keep_default_na=False, na_values=[])
                for _, row in df_old.iterrows():
                    k = (str(row.get("hotel_name", "")), str(row.get("room_type", "")))
                    written_keys.add(k)
                    if k in all_week_prices:
                        new_row = {"hotel_name": k[0], "room_type": k[1]}
                        for i in range(1, 7):
                            old_val = str(row.get(f"price_w{i}", "NA")).strip()
                            new_val = str(all_week_prices[k].get(f"Price W{i}", "NA")).strip()
                            if new_val in ("NA", "nan", "") and old_val not in ("NA", "nan", ""):
                                new_row[f"price_w{i}"] = old_val
                            else:
                                new_row[f"price_w{i}"] = new_val if new_val not in ("nan", "") else "NA"
                        rows.append(new_row)
                    else:
                        rows.append(row.to_dict())
            except:
                pass
        for (hotel, room), prices in all_week_prices.items():
            if (hotel, room) not in written_keys:
                row = {"hotel_name": hotel, "room_type": room}
                for i in range(1, 7):
                    row[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
                rows.append(row)
        tmp = filename + ".tmp"
        pd.DataFrame(rows).to_csv(tmp, index=False)
        os.replace(tmp, filename)
    except Exception as e:
        print(f"\u274c Error saving: {e}", flush=True)

def update_url_checkin(url, checkin_date):
    """Update Trip.com URL với checkIn & checkOut dates."""
    checkin_str = checkin_date.strftime("%Y-%m-%d")
    checkout_str = (checkin_date + timedelta(days=1)).strftime("%Y-%m-%d")
    # Replace existing checkIn/checkOut params
    if 'checkin=' in url.lower() or 'checkIn=' in url:
        url = re.sub(r'check[Ii]n=[\d-]+', f'checkIn={checkin_str}', url)
    else:
        url += f"{'&' if '?' in url else '?'}checkIn={checkin_str}"
    if 'checkout=' in url.lower() or 'checkOut=' in url:
        url = re.sub(r'check[Oo]ut=[\d-]+', f'checkOut={checkout_str}', url)
    else:
        url += f"&checkOut={checkout_str}"
    return url

def clean_price(raw):
    """Clean giá từ Trip.com — chỉ giữ 'VND x,xxx,xxx' hoặc NA/SOLD OUT."""
    if not raw or raw == "NA":
        return "NA"
    s = str(raw).strip()
    if s.startswith("SOLD OUT"):
        return s
    import re as _re
    m = _re.search(r'VND\s*([\d,]+)', s, _re.IGNORECASE)
    if m:
        return f"VND {m.group(1)}"
    m = _re.search(r'([\d,]+)\s*VND', s, _re.IGNORECASE)
    if m:
        return f"VND {m.group(1)}"
    m = _re.search(r'[\u20ab]\s*([\d,]+)|([\d,]+)\s*[\u20ab]', s)
    if m:
        num = m.group(1) or m.group(2)
        return f"VND {num}"
    m = _re.search(r'([\d,]{7,})', s)
    if m:
        return f"VND {m.group(1)}"
    return "NA"

def calc_na_stats(d):
    return sum(1 for i in range(1, 7) if d.get(f"Price W{i}", "NA") == "NA"), 6

def calc_batch_na_rate(awp, keys):
    tc, nc = 0, 0
    for k in keys:
        if k in awp:
            n, t = calc_na_stats(awp[k])
            nc += n; tc += t
    return nc / max(tc, 1), nc, tc

def find_retry_weeks(awp, keys):
    items = []
    for k in keys:
        if k not in awp: continue
        p = awp[k]
        has_real = any(v != "NA" and not str(v).startswith("SOLD OUT") for v in p.values())
        for i in range(1, 7):
            v = p.get(f"Price W{i}", "NA")
            if v == "NA" or (str(v).startswith("SOLD OUT") and not has_real):
                items.append((k, i))
    return items

def find_na_soldout_weeks(awp, keys):
    items = []
    for k in keys:
        if k not in awp: continue
        p = awp[k]
        for i in range(1, 7):
            v = p.get(f"Price W{i}", "NA")
            if v == "NA" or str(v).startswith("SOLD OUT"):
                items.append((k, i))
    return items

async def load_cookies_to_context(context):
    """Load cookies từ file vào Playwright browser context."""
    if not os.path.exists(COOKIES_FILE):
        print(f"\u26a0\ufe0f Cookies file not found: {COOKIES_FILE} — tiếp tục không cần login", flush=True)
        return
    with open(COOKIES_FILE, "r", encoding="utf-8") as f:
        cookies = json.load(f)
    # Playwright cookies format: name, value, domain, path, expires, httpOnly, secure, sameSite
    pw_cookies = []
    for c in cookies:
        cookie = {
            "name": c.get("name", ""),
            "value": c.get("value", ""),
            "domain": c.get("domain", ".trip.com"),
            "path": c.get("path", "/"),
        }
        if "expires" in c and c["expires"]:
            cookie["expires"] = c["expires"]
        elif "expiry" in c and c["expiry"]:
            cookie["expires"] = c["expiry"]
        if "httpOnly" in c:
            cookie["httpOnly"] = c["httpOnly"]
        if "secure" in c:
            cookie["secure"] = c["secure"]
        if "sameSite" in c:
            ss = c["sameSite"]
            if ss in ("Strict", "Lax", "None"):
                cookie["sameSite"] = ss
        pw_cookies.append(cookie)
    await context.add_cookies(pw_cookies)

# ============================================================
# CRAWL 1 NGÀY — thử 1 checkin date cụ thể (Trip.com)
# ============================================================
async def crawl_single_day(browser, hotel_url, room_type, week_num, checkin,
                           retries=None, page_timeout=None, hotel_name=""):
    if retries is None: retries = RETRIES_PER_DAY
    if page_timeout is None: page_timeout = PAGE_TIMEOUT

    result = {"week": week_num, "price": "NA", "date": checkin.strftime('%Y-%m-%d')}

    for retry in range(retries):
        context = None
        try:
            if retry > 0:
                backoff = random.uniform(3, 6) * (retry + 1)
                await asyncio.sleep(backoff)

            context = await browser.new_context(
                viewport={"width": random.randint(1366, 1920), "height": random.randint(768, 1080)},
                user_agent=random.choice(USER_AGENTS),
                locale="en-US",
            )
            # Load cookies vào context
            await load_cookies_to_context(context)

            page = await context.new_page()
            await page.add_init_script(STEALTH_SCRIPT)

            url = update_url_checkin(hotel_url, checkin)

            try:
                await page.goto(url, timeout=page_timeout, wait_until="domcontentloaded")
            except Exception:
                pass

            await asyncio.sleep(random.uniform(2, 4))

            # Scroll để trigger lazy load
            await page.evaluate("window.scrollTo({top: 300, behavior: 'smooth'})")
            await asyncio.sleep(random.uniform(0.5, 1.0))

            # Đóng popup nếu có
            try:
                close_btn = page.locator("div[class*='close'], button[class*='close'], .ab-close-button")
                if await close_btn.count() > 0:
                    await close_btn.first.click(timeout=2000)
            except: pass

            # Đợi room cards load
            try:
                await page.wait_for_selector("div[class*='commonRoomCard__']", timeout=15000)
            except:
                try:
                    await page.wait_for_selector("div[class*='saleRoomItemBox']", timeout=8000)
                    await asyncio.sleep(3)
                except:
                    await asyncio.sleep(3)

            # Scroll to load more rooms
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2)")
            await asyncio.sleep(0.5)
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(random.uniform(1.0, 2.0))

            extraction = await page.evaluate(EXTRACT_PRICES_JS, room_type)

            if extraction.get('found'):
                price = clean_price(extraction['price'])
                result["price"] = price
                if price != "NA":
                    return result

            if extraction.get('soldOut'):
                st = extraction.get('soldOutType', '')
                sp = extraction.get('soldOutPrice', '')
                price = "SOLD OUT" if st == 'hotel' else f"SOLD OUT {sp}".strip()
                result["price"] = price
                return result

            if retry == retries - 1:
                rooms = extraction.get('allRooms', 0)
                names = extraction.get('roomNames', [])[:3]
                result["debug_rooms"] = rooms
                result["debug_names"] = names

        except Exception as e:
            err = str(e)
            if "has been closed" in err or "Target page" in err:
                pass
        finally:
            if context:
                try: await context.close()
                except: pass

        await asyncio.sleep(random.uniform(*DELAY_RANGE))

    return result

# ============================================================
# CRAWL 1 TUẦN — thử từng ngày trong tuần cho đến khi có giá
# ============================================================
async def crawl_week_range(browser, hotel_url, room_type, week_num, base_checkin,
                           hotel_name="", days_in_week=None, page_timeout=None):
    if days_in_week is None: days_in_week = DAYS_PER_WEEK

    week_start_offset = (week_num - 1) * 7
    week_start = base_checkin + timedelta(days=week_start_offset)
    week_end = week_start + timedelta(days=days_in_week - 1)

    print(f"      \U0001f50d [{hotel_name[:20]}] W{week_num}: trying {week_start.strftime('%m/%d')}\u2192{week_end.strftime('%m/%d')} ...", flush=True)

    last_result = {"week": week_num, "price": "NA", "date": ""}
    has_sold_out = False
    sold_out_count = 0
    na_count = 0

    for day_offset in range(days_in_week):
        checkin = base_checkin + timedelta(days=week_start_offset + day_offset)
        result = await crawl_single_day(
            browser, hotel_url, room_type, week_num, checkin,
            page_timeout=page_timeout, hotel_name=hotel_name
        )

        price = result["price"]

        if price != "NA" and not str(price).startswith("SOLD OUT"):
            print(f"      \u2705 [{hotel_name[:20]}] W{week_num}: {price} | {checkin.strftime('%Y-%m-%d')} (day {day_offset+1}/{days_in_week})", flush=True)
            return result

        if str(price).startswith("SOLD OUT"):
            has_sold_out = True
            sold_out_count += 1
            last_result = result
        else:
            na_count += 1
            last_result = result

        if day_offset < days_in_week - 1:
            await asyncio.sleep(random.uniform(0.5, 1.5))

    if has_sold_out:
        last_result["price"] = "SOLD OUT"
        print(f"      \U0001f6ab [{hotel_name[:20]}] W{week_num}: SOLD OUT (tried {days_in_week} days: {sold_out_count} sold, {na_count} NA)", flush=True)
    else:
        print(f"      \u274c [{hotel_name[:20]}] W{week_num}: NA after trying all {days_in_week} days", flush=True)

    return last_result

# ============================================================
# PROCESS 1 HOTEL
# ============================================================
async def process_hotel(browser, hotel_info, prev_data, base_checkin, semaphore, awp=None):
    hotel_name, hotel_url, room_type = hotel_info
    key = (hotel_name, room_type)

    if key in prev_data:
        vals = [prev_data[key].get(f"Price W{i}", "NA") for i in range(1, 7)]
        all_real = all(v != "NA" and not str(v).startswith("SOLD OUT") for v in vals)
        if all_real:
            return key, prev_data[key], True

    async with semaphore:
        await asyncio.sleep(random.uniform(*HOTEL_DELAY))
        print(f"\n\U0001f3e8 {hotel_name} | {room_type}", flush=True)

        prices = {}
        weeks_to_crawl = []
        for wn in range(1, 7):
            kp = f"Price W{wn}"
            cached = prev_data[key].get(kp, "NA") if key in prev_data else "NA"
            if cached != "NA" and not str(cached).startswith("SOLD OUT"):
                prices[kp] = cached
            else:
                weeks_to_crawl.append(wn)

        if awp is not None:
            if key not in awp:
                awp[key] = {f"Price W{i}": "NA" for i in range(1, 7)}
            for kp, v in prices.items():
                awp[key][kp] = v

        if weeks_to_crawl:
            for wn in weeks_to_crawl:
                r = await crawl_week_range(
                    browser, hotel_url, room_type, wn, base_checkin,
                    hotel_name=hotel_name
                )
                prices[f"Price W{r['week']}"] = r["price"]
                if awp is not None:
                    awp[key][f"Price W{r['week']}"] = r["price"]
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                    print(f"      \U0001f4be [{hotel_name[:20]}] Saved W{r['week']} to temp", flush=True)

        na_c, _ = calc_na_stats(prices)
        so_c = sum(1 for i in range(1, 7) if str(prices.get(f"Price W{i}", "")).startswith("SOLD OUT"))
        icon = "\u2705" if na_c == 0 and so_c == 0 else f"\u26a0\ufe0f({na_c}NA)" if na_c else f"\U0001f6ab({so_c}SO)"
        print(f"   {icon} DONE: {hotel_name}", flush=True)
        return key, prices, False

# ============================================================
# RETRY
# ============================================================
async def retry_batch(browser, batch_infos, batch_keys, awp, base_checkin):
    ki = {(i[0], i[2]): i for i in batch_infos}
    for rn in range(1, MAX_RETRY_ROUNDS + 1):
        nr, nc, tc = calc_batch_na_rate(awp, batch_keys)
        if nr <= TARGET_NA_RATE: break
        items = find_retry_weeks(awp, batch_keys)
        if not items: break

        to = RETRY_PAGE_TIMEOUT[min(rn-1, len(RETRY_PAGE_TIMEOUT)-1)]
        print(f"\n\U0001f501 RETRY {rn}/{MAX_RETRY_ROUNDS} | {len(items)} cells", flush=True)
        await asyncio.sleep(random.uniform(*RETRY_COOL_DOWN))

        hna = {}
        for k, wn in items:
            hna.setdefault(k, []).append(wn)

        sem = asyncio.Semaphore(NUM_WORKERS)
        async def do_retry(k, weeks):
            async with sem:
                info = ki.get(k)
                if not info: return
                hn, hu, rt = info
                await asyncio.sleep(random.uniform(*HOTEL_DELAY))
                wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
                async def rw(wn):
                    async with wsem:
                        return await crawl_week_range(
                            browser, hu, rt, wn, base_checkin,
                            hotel_name=hn, page_timeout=to
                        )
                results = await asyncio.gather(*[rw(w) for w in weeks])
                for r in results:
                    if r["price"] != "NA":
                        awp[k][f"Price W{r['week']}"] = r["price"]
        await asyncio.gather(*[do_retry(k, w) for k, w in hna.items()])

# ============================================================
# AUTO RETRY NA & SOLD OUT
# ============================================================
async def auto_retry_na_soldout(browser, all_infos, awp, base_checkin):
    ki = {(i[0], i[2]): i for i in all_infos}
    all_keys = list(awp.keys())

    items = find_na_soldout_weeks(awp, all_keys)
    if not items:
        print(f"\n\u2705 Không có NA/SOLD OUT nào cần retry!", flush=True)
        return 0

    hna = {}
    for k, wn in items:
        hna.setdefault(k, []).append(wn)

    na_count = sum(1 for k, wn in items if awp[k].get(f"Price W{wn}", "NA") == "NA")
    so_count = sum(1 for k, wn in items if str(awp[k].get(f"Price W{wn}", "")).startswith("SOLD OUT"))

    print(f"\n{'='*60}", flush=True)
    print(f"\U0001f504 AUTO ROUND 2 \u2014 Re-crawl NA & SOLD OUT", flush=True)
    print(f"   \U0001f4ca {len(items)} cells ({na_count} NA + {so_count} SOLD OUT) across {len(hna)} hotels", flush=True)
    print(f"{'='*60}", flush=True)

    updated = 0
    sem = asyncio.Semaphore(NUM_WORKERS)

    async def do_hotel_retry(k, weeks):
        nonlocal updated
        async with sem:
            info = ki.get(k)
            if not info: return
            hn, hu, rt = info

            await asyncio.sleep(random.uniform(*HOTEL_DELAY))
            print(f"\n   \U0001f3e8 [R2] {hn} | weeks: {weeks}", flush=True)

            wsem = asyncio.Semaphore(WEEKS_PER_HOTEL)
            async def rw(wn):
                async with wsem:
                    return await crawl_week_range(
                        browser, hu, rt, wn, base_checkin,
                        hotel_name=hn, page_timeout=45000
                    )
            results = await asyncio.gather(*[rw(w) for w in weeks])
            for r in results:
                new_price = r["price"]
                old_price = awp[k].get(f"Price W{r['week']}", "NA")
                if new_price != old_price:
                    if new_price != "NA":
                        awp[k][f"Price W{r['week']}"] = new_price
                        old_label = "NA" if old_price == "NA" else "SO"
                        new_label = new_price if not str(new_price).startswith("SOLD OUT") else "SOLD OUT"
                        print(f"      \U0001f504 W{r['week']}: {old_label} \u2192 {new_label}", flush=True)
                        updated += 1

    await asyncio.gather(*[do_hotel_retry(k, w) for k, w in hna.items()])

    print(f"\n{_LINE*50}", flush=True)
    print(f"   \U0001f504 Round 2 complete: {updated}/{len(items)} cells updated", flush=True)
    print(f"{_LINE*50}", flush=True)

    return updated

# ============================================================
# MAIN
# ============================================================
async def main():
    t0 = time.time()

    df = read_hotels_from_source()
    if len(df) == 0: return

    awp, prev = {}, {}

    def load_prev(fp):
        n = 0
        try:
            dp = pd.read_csv(fp, keep_default_na=False, na_values=[])
            for _, row in dp.iterrows():
                k = (row["hotel_name"], row["room_type"])
                if k in prev:
                    for i in range(1, 7):
                        v = str(row.get(f"price_w{i}", "NA")).strip()
                        if v and v not in ("NA", "nan"):
                            prev[k][f"Price W{i}"] = v
                else:
                    p = {}
                    for i in range(1, 7):
                        v = str(row.get(f"price_w{i}", "NA")).strip()
                        p[f"Price W{i}"] = "NA" if (not v or v in ("nan", "NA")) else v
                    prev[k] = p; awp[k] = p
                n += 1
        except: pass
        return n

    if os.path.exists(TEMP_OUTPUT_FILE):
        n = load_prev(TEMP_OUTPUT_FILE)
        print(f"\U0001f4c2 Loaded {n} hotels from temp", flush=True)

    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=CHECKIN_OFFSET)
    all_infos = [(r['hotel_name'], r['hotel_url'], r['room_type']) for _, r in df.iterrows()]

    # Phân loại hotels
    new_hotels = []
    incomplete_hotels = []
    complete_hotels = []
    for info in all_infos:
        key = (info[0], info[2])
        if key not in prev:
            new_hotels.append(info)
        else:
            vals = [prev[key].get(f"Price W{i}", "NA") for i in range(1, 7)]
            all_real = all(v != "NA" and not str(v).startswith("SOLD OUT") for v in vals)
            if all_real:
                awp[key] = prev[key]
                complete_hotels.append(info)
            else:
                na_c = sum(1 for v in vals if v == "NA")
                so_c = sum(1 for v in vals if str(v).startswith("SOLD OUT"))
                incomplete_hotels.append((info, na_c, so_c))

    infos = new_hotels + [item[0] for item in incomplete_hotels]
    total = len(infos)

    print(f"\n\U0001f4ca Phân loại:", flush=True)
    print(f"   \U0001f195 {len(new_hotels)} hotel mới (crawl trước)", flush=True)
    print(f"   \U0001f504 {len(incomplete_hotels)} hotel chưa đủ (crawl NA/SOLD OUT)", flush=True)
    print(f"   \u2705 {len(complete_hotels)} hotel đã đủ 6w (skip)", flush=True)
    if complete_hotels:
        for info in complete_hotels[:5]:
            print(f"      \u23ed\ufe0f {info[0]}", flush=True)
        if len(complete_hotels) > 5:
            print(f"      ... và {len(complete_hotels) - 5} hotel khác", flush=True)

    print(f"\n\U0001f4c5 Week date ranges (try each day until price found):", flush=True)
    for wn in range(1, 7):
        w_start = bc + timedelta(days=(wn - 1) * 7)
        w_end = w_start + timedelta(days=DAYS_PER_WEEK - 1)
        print(f"   W{wn}: {w_start.strftime('%Y-%m-%d (%a)')} \u2192 {w_end.strftime('%Y-%m-%d (%a)')}", flush=True)

    print(f"\n{'='*60}", flush=True)
    print(f"\U0001f3ad CRAWL Trip.com v1 \u2014 Playwright async", flush=True)
    print(f"\U0001f4ca {total} hotels | {NUM_WORKERS}W \u00d7 {WEEKS_PER_HOTEL}wk | {DAYS_PER_WEEK} days/week", flush=True)
    print(f"\U0001f504 Auto Round 2: {'ON' if AUTO_RETRY_NA_SOLDOUT else 'OFF'}", flush=True)
    print(f"\U0001f4e1 Source: {'Google Sheet' if GOOGLE_SHEET_ID else 'Local file'}", flush=True)
    print(f"{'='*60}\n", flush=True)

    async with async_playwright() as p:
        if HEADLESS:
            browser = await p.chromium.launch(
                headless=False,
                args=[
                    '--headless=new',
                    '--disable-blink-features=AutomationControlled',
                    '--no-sandbox',
                    '--disable-dev-shm-usage',
                ],
            )
            print("\u2705 Browser launched (Chrome New Headless)", flush=True)
        else:
            browser = await p.chromium.launch(
                headless=False,
                args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-dev-shm-usage'],
            )
            print("\u2705 Browser launched (visible window)", flush=True)

        tb = (total + BATCH_SIZE - 1) // BATCH_SIZE
        ct = 0

        for bi in range(tb):
            bs, be = bi * BATCH_SIZE, min((bi+1) * BATCH_SIZE, total)
            batch = infos[bs:be]

            print(f"\n{'='*60}", flush=True)
            print(f"\U0001f4e6 BATCH {bi+1}/{tb} | Hotels {bs+1}-{be}/{total}", flush=True)
            print(f"{'='*60}", flush=True)

            sem = asyncio.Semaphore(NUM_WORKERS)
            tasks = [process_hotel(browser, i, prev, bc, sem, awp=awp) for i in batch]

            bkeys = []
            for coro in asyncio.as_completed(tasks):
                try:
                    k, prices, skip = await coro
                    awp[k] = prices; bkeys.append(k)
                    if not skip: ct += 1
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                except Exception as e:
                    print(f"\u274c {e}", flush=True)

            nr, nc, tc = calc_batch_na_rate(awp, bkeys)
            print(f"\n\U0001f4ca Batch {bi+1}: NA = {nr:.1%} ({nc}/{tc})", flush=True)
            if nr > TARGET_NA_RATE:
                await retry_batch(browser, batch, bkeys, awp, bc)
                save_backup_csv(awp, TEMP_OUTPUT_FILE)

            print(f"\n{_LINE*50}", flush=True)
            for k in bkeys:
                pr = awp[k]; parts = []
                for i in range(1, 7):
                    v = pr.get(f"Price W{i}", "NA")
                    parts.append("\u2713" if v != "NA" and not str(v).startswith("SOLD OUT") else "\U0001f6ab" if str(v).startswith("SOLD OUT") else "\u2717")
                nac, _ = calc_na_stats(pr)
                print(f"   {_CHECK if nac==0 else _WARN} {k[0][:35]:35s} {' '.join(parts)}", flush=True)
            print(f"{_LINE*50} | \u23f1\ufe0f {int((time.time()-t0)//60)}m | {ct}/{total}", flush=True)

        fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
        save_backup_csv(awp, fn)
        print(f"\n\U0001f4c1 Round 1 saved: {fn}", flush=True)

        tc_r1 = len(awp) * 6
        na_r1 = sum(1 for p in awp.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
        so_r1 = sum(1 for p in awp.values() for i in range(1,7) if str(p.get(f"Price W{i}","")).startswith("SOLD OUT"))
        print(f"   Round 1: \u2705 {tc_r1-na_r1-so_r1}/{tc_r1} | \U0001f6ab {so_r1} SO | \u274c {na_r1} NA", flush=True)

        if AUTO_RETRY_NA_SOLDOUT and (na_r1 > 0 or so_r1 > 0):
            r2_updated = await auto_retry_na_soldout(browser, infos, awp, bc)

            if r2_updated > 0:
                save_backup_csv(awp, fn)
                save_backup_csv(awp, TEMP_OUTPUT_FILE)
                print(f"\n\U0001f4c1 Final updated: {fn} ({r2_updated} cells changed)", flush=True)
            else:
                print(f"\n\U0001f4c1 Final unchanged (no improvements in Round 2)", flush=True)

        await browser.close()

    tt = time.time() - t0; tc = len(awp) * 6
    na_t = sum(1 for p in awp.values() for i in range(1,7) if p.get(f"Price W{i}","NA") == "NA")
    so_t = sum(1 for p in awp.values() for i in range(1,7) if str(p.get(f"Price W{i}","")).startswith("SOLD OUT"))
    print(f"\n{'='*60}", flush=True)
    print(f"\u2705 FINAL COMPLETED | {fn}", flush=True)
    print(f"   \u2705 Price: {tc-na_t-so_t}/{tc} ({(tc-na_t-so_t)/max(tc,1):.1%})", flush=True)
    print(f"   \U0001f6ab Sold:  {so_t}/{tc} | \u274c NA: {na_t}/{tc}", flush=True)
    print(f"\u23f1\ufe0f {int(tt//60)}m {int(tt%60)}s", flush=True)
    print(f"{'='*60}", flush=True)

await main()

✅ 20 hotels từ ./tr4.csv

📊 Phân loại:
   🆕 20 hotel mới (crawl trước)
   🔄 0 hotel chưa đủ (crawl NA/SOLD OUT)
   ✅ 0 hotel đã đủ 6w (skip)

📅 Week date ranges (try each day until price found):
   W1: 2026-03-05 (Thu) → 2026-03-11 (Wed)
   W2: 2026-03-12 (Thu) → 2026-03-18 (Wed)
   W3: 2026-03-19 (Thu) → 2026-03-25 (Wed)
   W4: 2026-03-26 (Thu) → 2026-04-01 (Wed)
   W5: 2026-04-02 (Thu) → 2026-04-08 (Wed)
   W6: 2026-04-09 (Thu) → 2026-04-15 (Wed)

🎭 CRAWL Trip.com v1 — Playwright async
📊 20 hotels | 6W × 7wk | 7 days/week
🔄 Auto Round 2: ON
📡 Source: Local file

✅ Browser launched (Chrome New Headless)

📦 BATCH 1/2 | Hotels 1-10/20

🏨 Melia Hanoi | Deluxe Room
      🔍 [Melia Hanoi] W1: trying 03/05→03/11 ...

🏨 Novotel Hanoi Thai Ha | Superior Room, 1 King Size Bed, City View
      🔍 [Novotel Hanoi Thai H] W1: trying 03/05→03/11 ...

🏨 Pan Pacific Hanoi | Deluxe King Or Twin Room
      🔍 [Pan Pacific Hanoi] W1: trying 03/05→03/11 ...

🏨 Movenpick Hotel Hanoi Centre | Classic Twin/Kin